In [1]:
%pip install yt-dlp faster-whisper

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 4.9 MB/s  0:00:00 eta 0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 8.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.5/39.5 MB 8.7 MB/s  0:00:04 eta 0:00:010:01:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 9.0 MB/s  0:00:029.5 MB/s eta 0:00:01:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 8.9 MB/s  0:00:04 eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [faster-whisper]━━━ 5/7 [onnxruntime]

[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


# 2.2 Q3 & Q4 — Building and evaluating the pipeline on real lectures

To test chunking on real speech rather than a synthetic example, I use two lectures that differ in useful ways:

1. **Andrej Karpathy — "[1hr Talk] Intro to Large Language Models"** (first 20 minutes): a clear speaker, with chapter timestamps in the description that serve as human-labelled topic boundaries.
2. **NPTEL "Environmental Degradation of Materials", Mod-01 Lec-07: Pourbaix diagram** (Dr. Kallol Mondal, IIT Kanpur; first 20 minutes): real classroom speech with heavy electrochemistry jargon.

In [5]:
from pathlib import Path
import json
import yt_dlp

LECTURE_DIR = Path("lectures")
LECTURE_DIR.mkdir(exist_ok=True)

lectures = {
    "karpathy_llm_intro": "https://www.youtube.com/watch?v=zjkBMFhNj_g",
       "nptel_pourbaix_lec07": "https://www.youtube.com/watch?v=3PDqGO39F8o",
}
CLIP_SECONDS = 20 * 60

def download(name, url):
    opts = {
        "format": "bestaudio/best",
        "outtmpl": str(LECTURE_DIR / f"{name}.%(ext)s"),
        "download_ranges": yt_dlp.utils.download_range_func(None, [(0, CLIP_SECONDS)]),
        "force_keyframes_at_cuts": True,
        "postprocessors": [{"key": "FFmpegExtractAudio", "preferredcodec": "mp3"}],
        "quiet": True,
    }
    with yt_dlp.YoutubeDL(opts) as ydl:
        info = ydl.extract_info(url, download=True)

    meta = {
        "title": info.get("title"),
        "duration": info.get("duration"),
        "chapters": info.get("chapters") or [],
    }
    (LECTURE_DIR / f"{name}_meta.json").write_text(json.dumps(meta, indent=2))
    print(f"{name}: {meta['title']} | chapters found: {len(meta['chapters'])}")

for name, url in lectures.items():
    if (LECTURE_DIR / f"{name}.mp3").exists():
        print(f"{name}: already downloaded, skipping")
        continue
    try:
        download(name, url)
    except yt_dlp.utils.DownloadError as e:
        print(f"{name}: FAILED -> {e}")

karpathy_llm_intro: already downloaded, skipping
nptel_pourbaix_lec07: already downloaded, skipping


In [6]:
for name in lectures:
    meta = json.loads((LECTURE_DIR / f"{name}_meta.json").read_text())
    print(f"\n{name} — {meta['title']}")
    for ch in meta["chapters"]:
        if ch["start_time"] < CLIP_SECONDS:
            print(f"  {ch['start_time'] / 60:5.1f} min  {ch['title']}")


karpathy_llm_intro — [1hr Talk] Intro to Large Language Models
    0.0 min  Intro: Large Language Model (LLM) talk
    0.3 min  LLM Inference
    4.3 min  LLM Training
    9.0 min  LLM dreams
   11.4 min  How do they work?
   14.2 min  Finetuning into an Assistant
   17.9 min  Summary so far

nptel_pourbaix_lec07 — Mod-01 Lec-07 Pourbaix diagram
    0.0 min  Potential pH diagram basics
    2.1 min  Nickel reduction reaction
    6.7 min  Nickel hydroxide formation
   11.8 min  Standard reduction potential
   17.6 min  Interpreting reaction lines


## Transcription with Whisper

I transcribe both clips with `faster-whisper` (a faster reimplementation of OpenAI's Whisper), keeping **word-level timestamps**. Timestamps are needed later for two things: detecting pauses as potential sentence boundaries, and matching chunks against the chapter boundaries.

In [7]:
from faster_whisper import WhisperModel

whisper = WhisperModel("small", device="cpu", compute_type="int8")

def transcribe(name):
    out_path = LECTURE_DIR / f"{name}_transcript.json"
    if out_path.exists():
        print(f"{name}: transcript exists, loading")
        return json.loads(out_path.read_text())

    segments, info = whisper.transcribe(
        str(LECTURE_DIR / f"{name}.mp3"),
        word_timestamps=True,
        vad_filter=True,
    )
    rows = []
    for seg in segments:
        rows.append({
            "start": seg.start,
            "end": seg.end,
            "text": seg.text.strip(),
            "words": [{"start": w.start, "end": w.end, "word": w.word} for w in seg.words],
        })
        print(f"[{seg.start / 60:5.1f} min] {seg.text.strip()}")

    out_path.write_text(json.dumps(rows, indent=2))
    return rows

transcripts = {name: transcribe(name) for name in lectures}

/home/sabari/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[  0.0 min] Hi, everyone. So recently, I gave a 30 minute talk on large language models, just kind of like
[  0.1 min] an intro talk. Unfortunately, that talk was not recorded, but a lot of people came to me after
[  0.2 min] the talk and they told me that they really liked the talk. So I thought I was just rerecorded
[  0.2 min] and basically put it up on YouTube. So here we go, the busy person's intro to large language
[  0.3 min] models, director Scott. Okay, so let's begin. First of all, what is a large language model
[  0.4 min] really? Well, a large language model is just two files, right? There will be two files in this
[  0.5 min] hypothetical directory. So for example, working with the specific example of the Lama270B model,
[  0.6 min] this is a large language model released by MetaAI. And this is basically the Lama series of
[  0.7 min] language models, the second iteration of it. And this is the 70 billion parameter model
[  0.8 min] of this series. So there's multiple mode

In [8]:
def full_text(rows):
    return " ".join(r["text"] for r in rows).lower()

checks = {
    "nptel_pourbaix_lec07": ["pourbaix", "pore-based", "nernst", "norse", "loss equation"],
    "karpathy_llm_intro": ["reversal curse", "reversal course", "chatgpt", "chat gpt", "chart apt"],
}

for name, terms in checks.items():
    text = full_text(transcripts[name])
    print(f"\n{name}")
    for t in terms:
        print(f"  {t!r:20} appears {text.count(t)} times")


nptel_pourbaix_lec07
  'pourbaix'           appears 0 times
  'pore-based'         appears 4 times
  'nernst'             appears 0 times
  'norse'              appears 1 times
  'loss equation'      appears 1 times

karpathy_llm_intro
  'reversal curse'     appears 0 times
  'reversal course'    appears 1 times
  'chatgpt'            appears 0 times
  'chat gpt'           appears 3 times
  'chart apt'          appears 1 times


### What Whisper got wrong

Transcription errors are concentrated in exactly the words that matter most for retrieval: technical terms, names, and numbers.

| Said | Transcribed |
|---|---|
| Pourbaix diagram | "pore-based diagram" (every occurrence) |
| Nernst equation | "Norse equation" and "loss equation" |
| Ni²⁺ + 2e⁻ → Ni | "Ni++++ to E" |
| reversal curse | "reversal course" |
| 140 gigabytes | "104 gigabytes" (once; correct elsewhere) |
| ChatGPT | "chart APT" |

**Consequences for retrieval:**
- In a lecture entirely about Pourbaix diagrams, the word "Pourbaix" never appears in the transcript, so keyword search for it would return nothing.
- The same term is misrecognised inconsistently ("Norse" / "loss"), scattering one concept across several spellings.
- Misheard numbers ("104 GB") would be retrieved and stated confidently as fact.
- The lecturer corrects his own arithmetic mid-calculation ("sorry, I made a mistake"); a chunk ending before the correction contains a wrong value.
- The NPTEL lecture opens with "In the last lecture we have seen…", so its context depends on a different document entirely.
- **Measured:** "Pourbaix" appears 0 times in the NPTEL transcript while "pore-based" appears 4 times; "Nernst" appears 0 times, split between "Norse" and "loss equation". Even the spacing variant "chat gpt" (3 occurrences) means a keyword search for "ChatGPT" returns nothing.

## Q3 — Evaluating chunking against real topic boundaries

YouTube chapter timestamps serve as human-labelled topic boundaries. I chunk each real transcript three ways and measure:

- **Boundary hit rate:** the fraction of true chapter boundaries that have a chunk boundary within ±15 seconds.
- **Chunk purity:** the fraction of chunks that lie entirely within a single chapter (i.e. do not mix topics).

To keep the comparison fair, every method produces the **same number of chunks**, and each is compared against a **random baseline** that places the same number of cuts at random sentence boundaries.

In [9]:
import random
import numpy as np
import tiktoken
from sentence_transformers import SentenceTransformer

enc = tiktoken.get_encoding("cl100k_base")
embedder = SentenceTransformer("all-MiniLM-L6-v2")

def to_sentences(rows, pause=0.8):
    words = [w for r in rows for w in r["words"]]
    sents, cur = [], []
    for i, w in enumerate(words):
        cur.append(w)
        ends_sentence = w["word"].strip().endswith((".", "?", "!"))
        gap = words[i + 1]["start"] - w["end"] if i + 1 < len(words) else 999
        if ends_sentence or gap > pause:
            sents.append({"text": "".join(x["word"] for x in cur).strip(),
                          "start": cur[0]["start"], "end": cur[-1]["end"]})
            cur = []
    return sents

Loading weights: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 8381.28it/s]


In [10]:
def pack(items, max_tokens, text_of):
    groups, cur, n = [], [], 0
    for it in items:
        t = len(enc.encode(text_of(it)))
        if cur and n + t > max_tokens:
            groups.append(cur)
            cur, n = [], 0
        cur.append(it)
        n += t
    if cur:
        groups.append(cur)
    return [{"start": g[0]["start"], "end": g[-1]["end"]} for g in groups]

def fixed_chunks(rows, max_tokens):
    words = [w for r in rows for w in r["words"]]
    return pack(words, max_tokens, lambda w: w["word"])

def sentence_chunks(sents, max_tokens):
    return pack(sents, max_tokens, lambda s: s["text"])

def semantic_chunks(sents, n_chunks, block=3, min_sents=3):
    emb = embedder.encode([s["text"] for s in sents], normalize_embeddings=True)
    sims = []
    for g in range(1, len(sents)):
        left = emb[max(0, g - block):g].mean(axis=0)
        right = emb[g:g + block].mean(axis=0)
        sims.append(left @ right / (np.linalg.norm(left) * np.linalg.norm(right)))
    cuts = []
    for idx in np.argsort(sims):
        g = idx + 1
        if all(abs(g - c) >= min_sents for c in cuts):
            cuts.append(g)
        if len(cuts) == n_chunks - 1:
            break
    bounds = [0] + sorted(cuts) + [len(sents)]
    return [{"start": sents[a]["start"], "end": sents[b - 1]["end"]}
            for a, b in zip(bounds[:-1], bounds[1:])]

In [11]:
def evaluate(chunks, chapter_times, tol=15):
    cut_times = [c["start"] for c in chunks[1:]]
    hit = np.mean([any(abs(ct - b) <= tol for ct in cut_times) for b in chapter_times])
    chapter_of = lambda t: sum(t >= b for b in chapter_times)
    purity = np.mean([chapter_of(c["start"]) == chapter_of(c["end"]) for c in chunks])
    return hit, purity

def random_baseline(sents, n_chunks, chapter_times, runs=200):
    scores = []
    for _ in range(runs):
        cuts = sorted(random.sample(range(1, len(sents)), n_chunks - 1))
        bounds = [0] + cuts + [len(sents)]
        chunks = [{"start": sents[a]["start"], "end": sents[b - 1]["end"]}
                  for a, b in zip(bounds[:-1], bounds[1:])]
        scores.append(evaluate(chunks, chapter_times))
    return tuple(np.mean(scores, axis=0))

random.seed(0)
for name in lectures:
    meta = json.loads((LECTURE_DIR / f"{name}_meta.json").read_text())
    rows = transcripts[name]
    sents = to_sentences(rows)
    end = sents[-1]["end"]
    chapter_times = [ch["start_time"] for ch in meta["chapters"] if 0 < ch["start_time"] < end]
    print(f"\n{name}: {len(sents)} sentences, {len(chapter_times)} chapter boundaries")
    if not chapter_times:
        continue
    for size in [256, 512]:
        fixed = fixed_chunks(rows, size)
        k = len(fixed)
        results = {
            "fixed": evaluate(fixed, chapter_times),
            "sentence": evaluate(sentence_chunks(sents, size), chapter_times),
            "semantic": evaluate(semantic_chunks(sents, k), chapter_times),
            "random": random_baseline(sents, k, chapter_times),
        }
        print(f"  size={size} tokens, {k} chunks")
        for m, (hit, pur) in results.items():
            print(f"    {m:<9} boundary hit rate={hit:.2f}   purity={pur:.2f}")


karpathy_llm_intro: 225 sentences, 6 chapter boundaries
  size=256 tokens, 18 chunks
    fixed     boundary hit rate=0.67   purity=0.67
    sentence  boundary hit rate=0.33   purity=0.68
    semantic  boundary hit rate=0.50   purity=0.72
    random    boundary hit rate=0.41   purity=0.69
  size=512 tokens, 9 chunks
    fixed     boundary hit rate=0.50   purity=0.33
    sentence  boundary hit rate=0.33   purity=0.40
    semantic  boundary hit rate=0.33   purity=0.44
    random    boundary hit rate=0.22   purity=0.47

nptel_pourbaix_lec07: 84 sentences, 4 chapter boundaries
  size=256 tokens, 10 chunks
    fixed     boundary hit rate=0.25   purity=0.60
    sentence  boundary hit rate=0.50   purity=0.67
    semantic  boundary hit rate=0.00   purity=0.70
    random    boundary hit rate=0.22   purity=0.65
  size=512 tokens, 5 chunks
    fixed     boundary hit rate=0.25   purity=0.20
    sentence  boundary hit rate=0.00   purity=0.20
    semantic  boundary hit rate=0.00   purity=0.60
    ra

### Results

| Lecture | Size | Chunks | Method | Boundary hit rate | Purity |
|---|---|---|---|---|---|
| Karpathy | 256 | 18 | fixed | 0.67 | 0.67 |
| | | | sentence | 0.33 | 0.68 |
| | | | semantic | 0.50 | **0.72** |
| | | | *random* | *0.41* | *0.69* |
| Karpathy | 512 | 9 | fixed | 0.50 | 0.33 |
| | | | sentence | 0.33 | 0.40 |
| | | | semantic | 0.33 | 0.44 |
| | | | *random* | *0.22* | *0.47* |
| NPTEL | 256 | 10 | fixed | 0.25 | 0.60 |
| | | | sentence | 0.50 | 0.67 |
| | | | semantic | 0.00 | **0.70** |
| | | | *random* | *0.22* | *0.65* |
| NPTEL | 512 | 5 | fixed | 0.25 | 0.20 |
| | | | sentence | 0.00 | 0.20 |
| | | | semantic | 0.00 | **0.60** |
| | | | *random* | *0.09* | *0.43* |

**The random baseline is essential for reading these numbers.** Fixed-size chunking appears best at boundary detection on Karpathy (0.67), but random cuts score 0.41 at the same chunk count. With 18 cuts each covering a ±15 s window, roughly 45% of the 20-minute clip is "covered" by chance alone.

**Boundary hit rate cannot separate the methods on this data.** Karpathy has only 6 chapter boundaries (each worth 0.17) and NPTEL only 4. The difference between fixed (0.67) and random (0.41) amounts to roughly 4 versus 2.5 boundaries, well within chance. With 10 boundaries in total, no hit-rate difference here is meaningful.

**Semantic chunking shows a modest advantage in purity.** It produced the purest chunks in 3 of 4 settings, most clearly on NPTEL at 512 tokens (0.60, versus 0.43 random and 0.20 fixed). It is better at *not mixing topics* than at *locating boundaries precisely*.

**The NPTEL lecture exposed the run-on problem from Q1.** Karpathy's 20 minutes produced 225 sentences (~5 s each), NPTEL's only 84 (~14 s each): the lecturer derives equations continuously with few pauses or sentence ends. With such coarse units, no chunker can place cuts precisely.

**Limitations of this evaluation:**
- Chapter timestamps mark where the uploader placed labels, often at slide changes, which need not coincide with the moment the spoken topic shifts.
- The ±15 s tolerance is strict relative to 5–14 s sentences.
- The origin of the NPTEL chapters (human-written or auto-generated) is unverified.
- Two lectures and 10 boundaries are too few for reliable conclusions.

**Conclusion:** on real lectures, semantic chunking kept topics less mixed, but no method reliably beat random at locating topic boundaries, and the evaluation itself is underpowered. Verifying that context loss is actually reduced needs a stronger design, described below.

## Q3 — Proposed pipeline for the full archive (~40,000 hours)

Each stage below responds to a failure observed in Q1, Q2, or the real-lecture experiments above.

**1. Transcribe with word-level timestamps.** Run Whisper (e.g. `faster-whisper`) with word timestamps and voice-activity detection. Timestamps enable pause-based segmentation and let users jump to the exact moment in the original audio. Because ASR errors concentrate on technical terms ("Pourbaix" → "pore-based"), each course would get a small **vocabulary prompt** (Whisper's `initial_prompt`) listing its key terms. I did not test this here; it is future work.

**2. Segment into sentences using punctuation *and* pauses.** Whisper's punctuation is unreliable on continuous speech, so a pause longer than ~0.8 s also ends a sentence. As the NPTEL lecture showed (~14 s per "sentence" versus ~5 s for Karpathy), some speakers still produce long run-ons, so over-long sentences should be split further at shorter pauses.

**3. Keep two versions of the text.** Embed a cleaned copy (filler words removed) so filler does not dilute the embedding, but store and display the original transcript, since an archive should not silently alter its record.

**4. Detect topic boundaries by comparing non-overlapping blocks of sentences** (TextTiling-style). Q2 showed that single sentences lose pronoun context, while overlapping windows smear boundaries; comparing blocks either side of each gap addresses both.

**5. Enforce minimum and maximum chunk sizes.** Merge fragments smaller than a few sentences into their neighbours and split any chunk exceeding the embedding model's input limit (e.g. MiniLM truncates at 256 word pieces, silently ignoring the rest).

**6. Attach context beyond the cut.** Q2 showed that a topic's introduction and its explanation may be non-adjacent, which no contiguous chunking can fix. Each chunk is therefore embedded with a short header (lecture title and topic label), and retrieval returns the matched chunk *plus its neighbours* (parent-child retrieval), so the model sees the surrounding context.

**7. Store with metadata.** Each chunk's vector is stored alongside its lecture ID, start/end timestamps, speaker, and language.

**8. Use hybrid retrieval.** Combine embedding search (robust to rephrasing) with keyword search (BM25, precise on exact names and numbers). Neither alone is sufficient: keyword search fails completely when ASR mangles a term (0 occurrences of "Pourbaix"), while embeddings can miss exact identifiers.

**Rough scale (order-of-magnitude estimates).** At ~150 words per minute, 40,000 hours is roughly 360 million words, about 480 million tokens, or ~2 million chunks at ~250 tokens each. At 384 dimensions × 4 bytes, that is only ~3 GB of vectors. Boundary detection needs roughly 27 million sentence embeddings, which a small model like MiniLM can process in hours on a GPU. The dominant cost is transcription itself, so generating LLM summaries for every chunk would be expensive; context headers should be generated per section, not per chunk.

## Q3 — A stronger evaluation

My chapter-based evaluation could not reliably separate the methods: with only 10 boundaries, differences were within chance, and chapter timestamps are an imperfect proxy for where spoken topics change. Verifying that context loss is actually reduced needs a design that measures it directly:

**1. Build an answer-span test set.** Sample passages from many lectures and have an LLM generate a question for each, together with the exact timestamps of the passage that answers it. A human verifies a random sample for correctness. Deliberately include **boundary-straddling questions**, whose answers cross a point where fixed-size chunking would cut, since these are what separate the methods.

**2. Measure context loss directly:**
- **Split rate:** the fraction of answer spans cut by at least one chunk boundary. This is the most direct measure of "context loss across boundaries" and needs no retrieval.
- **Full-answer Recall@k:** the fraction of questions for which a top-k retrieved chunk contains the *entire* answer span, not just its introduction (the exact failure in the question).
- **Orphan rate:** the fraction of chunks beginning with an unresolved reference ("it", "this", "so as I said"), a cheap proxy that needs no labels.

**3. Keep the comparison fair.** Compare methods at equal chunk counts and report the total tokens sent to the LLM, since larger chunks trivially contain more complete answers.

**4. Use enough data and report uncertainty.** Hundreds of lectures and around a thousand questions, with bootstrap 95% confidence intervals, so small differences are not mistaken for real ones, as they could have been in my 10-boundary experiment.

**5. Break results down by lecture type.** Report clean speech and jargon-heavy classroom lectures (like NPTEL) separately, since the real-lecture results suggest methods behave differently on them.

## Q4 (bonus) — Making the lectures queryable

Both transcripts are chunked with the semantic (block-comparison) chunker, and each chunk is embedded with a short context header (the lecture title), as proposed in step 6 of the pipeline. A question is embedded the same way, and the most similar chunks are returned with their timestamps.

The test questions are chosen to probe the failures found above, especially whether retrieval survives ASR errors: the transcript never contains "Pourbaix" or "reversal curse".

In [12]:
def chunk_texts(chunks, sents):
    out = []
    for c in chunks:
        text = " ".join(s["text"] for s in sents
                        if s["start"] >= c["start"] and s["end"] <= c["end"])
        out.append({**c, "text": text})
    return out

index = []
for name in lectures:
    meta = json.loads((LECTURE_DIR / f"{name}_meta.json").read_text())
    sents = to_sentences(transcripts[name])
    k = len(fixed_chunks(transcripts[name], 256))
    for c in chunk_texts(semantic_chunks(sents, k), sents):
        index.append({**c, "lecture": meta["title"], "name": name})

vectors = embedder.encode([f"Lecture: {c['lecture']}. {c['text']}" for c in index],
                          normalize_embeddings=True)
print(len(index), "chunks indexed")

28 chunks indexed


In [13]:
def ask(question, k=3):
    q = embedder.encode([question], normalize_embeddings=True)[0]
    scores = vectors @ q
    print(f"\nQ: {question}")
    for i in np.argsort(-scores)[:k]:
        c = index[i]
        print(f"  [{scores[i]:.2f}] {c['name']} @ {c['start']/60:.1f}–{c['end']/60:.1f} min")
        print(f"      {c['text'][:250]}...")

ask("What is the reversal curse?")
ask("How much does it cost to train Llama 2?")
ask("What is a Pourbaix diagram?")
ask("Why is the nickel line horizontal on the potential-pH diagram?")


Q: What is the reversal curse?
  [0.31] karpathy_llm_intro @ 12.7–13.0 min
      So a recent viral example is what we call the reversal course. So as an example, if you go to chat gpt and you talk to gpt for the best language model currently available, you say, who is Tom Cruise's mother, it will tell you it's merely Pfeiffer, wh...
  [0.14] nptel_pourbaix_lec07 @ 0.2–0.3 min
      In the last lecture we have seen a...
  [0.13] nptel_pourbaix_lec07 @ 12.1–12.3 min
      Now that case I can find it out, let me remove this part I can find out,...

Q: How much does it cost to train Llama 2?
  [0.26] karpathy_llm_intro @ 3.4–3.9 min
      So this is how we can run the model, just requires two files, just requires a MacBook. I'm slightly cheating here because this was not actually in terms of the speed of this video here, this was not running a 70 billion parameter model, it was only r...
  [0.24] karpathy_llm_intro @ 3.9–5.4 min
      But the computational complexity really comes in when 

### Results

| Question | Top result correct? | What happened |
|---|---|---|
| What is the reversal curse? | ✅ Yes | Found despite the transcript saying "reversal **course**" (0 exact matches for "reversal curse") |
| How much does it cost to train Llama 2? | ⚠️ 2nd | The answer ("$2 million") is at the very end of a 1.5-minute chunk; scores for all three results were nearly identical (0.23–0.26) |
| What is a Pourbaix diagram? | ❌ False success | High scores (0.70), but the top chunks were meaningless fragments ("In the last lecture we have seen a…") |
| Why is the nickel line horizontal? | ❌ No | The explanation (~6.5 min) is inside a 7.8-minute chunk, ranked 2nd |

**Embedding search is robust to some ASR errors.** The reversal-curse question succeeded even though the transcript contains "reversal course", where keyword search would have returned nothing.

**The context header dominated tiny chunks.** Every chunk was embedded with the lecture title ("…Pourbaix diagram"). For fragments of only a few words, the header *is* essentially the whole embedding, so the shortest, least informative chunks scored highest for "What is a Pourbaix diagram?". The fragments came from the lecturer's hesitation pauses ("let me remove this part… I can find out"), which the 0.8 s pause rule treated as sentence ends, exactly the failure predicted for Strategy A in Q2. A minor bug also contributed: the minimum-spacing rule kept cuts apart from each other, but not from the start and end of the transcript.

**Oversized chunks hide their content.** The semantic chunker has no maximum size and produced a 7.8-minute chunk. MiniLM reads only the first 256 word pieces of any input, so the explanation at ~6.5 min was invisible to search. The Llama cost answer likely suffered the same way, sitting at the end of a 1.5-minute chunk.

**Takeaway:** retrieval handled rephrasing and some transcription errors, but chunk size broke it in both directions: fragments too small to carry meaning, and chunks too large to be fully read. This is direct evidence for step 5 of the proposed pipeline (enforcing both minimum and maximum chunk sizes), and shows that a context header must be weighed against chunk length rather than simply prepended.